In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *

# UCLA sessions: checks, then compile

UCLA tested 17 patients (OFF and ON medication) with the same task as the other sites. `raw/ucla_data/` holds

- `bradykinetic-dominant/` and `tremor-dominant/`: one MAT file per session, e.g. `Brady_CG_OFFmeds_rl.mat`, and for
  eight patients an `..._eq.mat` file per session: about 225 trials at 50:50 on both colours, run immediately before
  the prior block. **The `eq` file is the session's practice** and is compiled as its practice block;
- `explicit/`: extra OFF sessions of six patients, compiled as separate sessions (`CG_off_explicit`), again with their
  `eq` file as practice where there is one;
- `hc/`: empty;
- `Perugini_2016/`: the Perugini et al. (2016) dataset. **UCLA's healthy controls are its 10 key-press controls**
  (`Reports_HC_Exp2_KeyPress/*_TrialByTrial_*.mat`, as notebook 0.01 compiles them), compiled here with the patients;
  its 4 eye-movement controls are compiled apart and not used (config `INCLUDE_EYE_TRACKING_SESSIONS: false`).

As at Case Western there are no block files, so the checks are those for whole-file sessions
(`src/utils/blocks.py`): header times, one continuous clock per file and its pauses, the design over the session in
160-trial stretches, practice ending before its session on the clock, and 600+ trials. Sessions are named by patient,
medication state and (for explicit sessions) instruction only; the subtype folders and prefixes and the prior labels
in the raw names are not carried over.

Writes `compiled/ucla/{blocks.csv, session_summary.csv, sessions/<session>.parquet}` (patients and key-press
controls) and `compiled/ucla/perugini/{session_summary.csv, sessions/<session>.parquet}` (eye-movement controls).

In [2]:
import re
from collections import Counter

import scipy.io

from src.data_io import catalog
from src.utils import blocks as blk
from src.utils.sessions import decode_perugini_eye, decode_perugini_keypress

UCLA = catalog.path("raw_ucla_sessions")
PERUGINI = catalog.path("raw_perugini")
SESSION_FOLDERS = ("bradykinetic-dominant", "tremor-dominant", "explicit")
SESSION_FILE = re.compile(r"^(?:(?P<subtype>Brady|Trem)_)?(?P<subject>[A-Z]+)_(?P<med>ON|OFF)(?P<kind>meds|expl|medsexpl)?_(?P<label>[a-z]{2})$")

## 1. Inventory

**Raw changes on 2026-10-20.** `tremor-dominant/Trem_DCAM_OFFmeds_rr.mat` was restored from
`raw_backup_20261019/DCAM_OFFmeds_rr.mat` (it had gone missing in the reorganisation of 2026-10-19; the earlier
pipeline used it), and `explicit/Brady_SGA_OFFmeds_practice.mat`, a byte-identical copy of Case Western's
`practice/Brady_SGA_OFFmedspractice.mat` (SGA is a Case Western patient), was removed. Both are in the backup.

In [3]:
listing, labels, unrecognised = [], [], []
for folder in SESSION_FOLDERS:
    for path in sorted((UCLA / folder).iterdir()):
        found = SESSION_FILE.match(path.stem) if path.suffix == ".mat" else None
        if not found:
            unrecognised.append(path.relative_to(UCLA).as_posix())
            continue
        explicit = "expl" in (found["kind"] or "")
        if explicit != (folder == "explicit"):
            raise ValueError(f"{path.name}: explicit name outside explicit/ or the reverse")
        medication = found["med"].lower()
        session_id = f"{found['subject']}_{medication}" + ("_explicit" if explicit else "")
        phase = "practice" if found["label"] == "eq" else "trial"
        listing.append({"session_id": session_id, "subject_id": found["subject"], "group": "pd", "medication": medication, "phase": phase,
                        "file": f"{session_id}/{phase}", "path": path.relative_to(UCLA).as_posix()})
        labels.append({"file": f"{session_id}/{phase}", "name_label": found["label"]})
listing = pd.DataFrame(listing, columns=blk.LISTING_COLUMNS)
assert not listing.file.duplicated().any(), "two raw files for one session block"

blocks, decoded = blk.read_session_files(listing, UCLA)
sessions_listed = blocks[blocks.phase == "trial"]
print(f"{len(blocks)} files: {len(sessions_listed)} sessions ({sessions_listed.session_id.str.endswith('_explicit').sum()} explicit) of "
      f"{blocks.subject_id.nunique()} patients, {(blocks.phase == 'practice').sum()} practice (eq) files")
print("not session files:", unrecognised or "none")
print("hc/:", sorted(p.name for p in (UCLA / "hc").iterdir()) or "empty")
print("MAT versions:", dict(Counter(blocks.mat_version)))
missing = [s for s in blocks.subject_id.unique() if {f"{s}_off", f"{s}_on"} - set(sessions_listed.session_id)]
print("patients without both OFF and ON regular sessions:", missing or "none")
blocks[["session_id", "phase", "header_time", "mat_version", "n_rows", "duration_min", "n_pauses", "longest_pause_min", "clock_backwards", "valid_pct"]].round(1)

60 files: 40 sessions (6 explicit) of 17 patients, 20 practice (eq) files
not session files: none
hc/: empty
MAT versions: {'5': 15, '7.3': 45}
patients without both OFF and ON regular sessions: none


,session_id,phase,header_time,mat_version,n_rows,duration_min,n_pauses,longest_pause_min,clock_backwards,valid_pct
0,AJL_off,trial,2022-11-14 10:14:23,5,731,28.9,0,0.0,0,83.4
1,AJL_on,trial,2022-11-14 10:14:23,5,663,30.2,0,0.0,0,89.4
2,CG_off,practice,2018-05-03 19:03:21,7.3,442,9.8,1,0.4,0,49.1
3,CG_off,trial,2018-05-03 19:03:21,7.3,1017,26.4,1,0.8,0,61.6
4,CG_on,practice,2018-05-03 19:03:21,7.3,226,9.5,0,0.0,0,96.9
5,CG_on,trial,2018-05-03 19:03:21,7.3,702,30.0,1,0.7,0,97.3
6,COH_off,practice,2018-05-03 19:03:21,7.3,227,9.7,0,0.0,0,93.4
7,COH_off,trial,2018-05-03 19:03:21,7.3,697,29.3,1,0.8,0,93.1
8,COH_on,practice,2018-05-03 19:03:21,7.3,237,9.7,0,0.0,0,92.0
9,COH_on,trial,2018-05-03 19:03:21,7.3,774,30.8,1,1.0,0,87.7


## 2. Header times

In [4]:
blocks["header_is_batch_save"] = blk.batch_saves(blocks.header_time)
for when, group in blocks.groupby(blocks.header_time.dt.floor("min")):
    print(f"{when:%Y-%m-%d %H:%M}: {len(group)} files" + (" (batch re-save)" if group.header_is_batch_save.all() else ""))
print(f"files with an original header: {(~blocks.header_is_batch_save).sum()}")

2018-05-03 19:03: 16 files (batch re-save)
2018-05-03 19:11: 29 files (batch re-save)
2022-11-14 10:14: 15 files (batch re-save)
files with an original header: 0


## 3. Design over the session

In [5]:
blk.add_design(blocks, decoded)
blk.add_stretch_priors(blocks, decoded)
blocks = blocks.merge(pd.DataFrame(labels), on="file")
practice = blocks[blocks.phase == "practice"]
print(f"practice (eq) files: {len(practice)}, coherences {dict(Counter(practice.coherences))}, priors {dict(Counter(practice.block_prior))}, "
      f"{practice.n_shown.min():.0f}-{practice.n_shown.max():.0f} trials")
blocks.loc[blocks.phase == "trial", ["session_id", "n_shown", "coherences", "green_right_pct", "red_right_pct", "block_prior", "design_error", "stretch_priors"]].round(1)

practice (eq) files: 20, coherences {'0,13,35,100': 20}, priors {'none': 20}, 223-285 trials


,session_id,n_shown,coherences,green_right_pct,red_right_pct,block_prior,design_error,stretch_priors
0,AJL_off,640,"0,13,35,100",75.0,50.0,gr,0.0,gr | gr | gr | gr
1,AJL_on,641,"0,13,35,100",49.8,25.0,rl,0.0,rl | rl | rl | rl
3,CG_off,640,"0,13,35,100",50.0,25.0,rl,0.0,rl | rl | rl | rl
5,CG_on,699,"0,13,35,100",24.9,50.0,gl,0.0,gl | gl | gl | gl
7,COH_off,664,"0,13,35,100",50.0,25.2,rl,0.0,rl | rl | rl | rl
9,COH_on,700,"0,13,35,100",50.1,24.8,rl,0.0,rl | rl | rl | rl
11,MBY_off,654,"0,13,35,100",49.7,75.0,rr,0.0,rr | rr | rr | rr
13,MBY_on,643,"0,13,35,100",24.9,50.3,gl,0.0,gl | gl | gl | gl
14,PAM_off,673,"0,13,35,100",75.1,50.0,gr,0.0,gr | gr | gr | gr
15,PAM_on,652,"0,13,35,100",49.5,25.1,rl,0.0,rl | rl | rl | rl


## 4. Session summary

Practice (the `eq` file) must end before its session starts on the clock; the gap is usually about a minute.

In [6]:
summary = blk.session_file_summary(blocks)
summary["instruction"] = np.where(summary.session_id.str.endswith("_explicit"), "explicit", "implicit")
summary[["session_id", "duration_min", "n_pauses", "n_shown", "valid_pct", "session_prior", "prior_consistent", "design_ok",
         "n_practice_blocks", "practice_to_trial_min", "issues"]].round(1)

,session_id,duration_min,n_pauses,n_shown,valid_pct,session_prior,prior_consistent,design_ok,n_practice_blocks,practice_to_trial_min,issues
0,AJL_off,28.9,0,640,83.4,gr,True,True,0,NaN,no practice file
1,AJL_on,30.2,0,641,89.4,rl,True,True,0,NaN,no practice file
2,CG_off,26.4,1,640,61.6,rl,True,True,1,1.5,
3,CG_on,30.0,1,699,97.3,gl,True,True,1,1.0,
4,COH_off,29.3,1,664,93.1,rl,True,True,1,2.1,
5,COH_on,30.8,1,700,87.7,rl,True,True,1,3.9,
6,MBY_off,30.1,1,654,94.2,rr,True,True,1,2.7,
7,MBY_on,27.0,0,643,96.3,gl,True,True,1,1.1,
8,PAM_off,30.7,0,673,86.5,gr,True,True,0,NaN,no practice file
9,PAM_on,29.1,0,652,91.7,rl,True,True,0,NaN,no practice file


## 5. Labels, counterbalancing and explicit sessions

The prior in each raw file name against the data (only reported here; names are not saved), whether each patient's
OFF and ON sessions used different priors, and the explicit sessions next to the same patient's regular OFF session.

In [7]:
trial_files = blocks[blocks.phase == "trial"].set_index("session_id")
label_mismatch = trial_files[trial_files.name_label != trial_files.block_prior]
print("file-name prior differs from the data:", ", ".join(f"{i} (name {r.name_label}, data {r.block_prior})" for i, r in label_mismatch.iterrows()) or "none")
regular = summary[summary.instruction == "implicit"].pivot(index="subject_id", columns="medication", values="session_prior")
same_prior = regular[regular.off == regular.on]
print("same prior in OFF and ON:", ", ".join(f"{s} ({p})" for s, p in same_prior.off.items()) or "none")
explicit = summary[summary.instruction == "explicit"].set_index("subject_id")
explicit_table = pd.DataFrame({"explicit_prior": explicit.session_prior, "regular_off_prior": regular.off.reindex(explicit.index),
                               "explicit_practice": explicit.n_practice_blocks, "explicit_trials": explicit.n_shown})
explicit_table

file-name prior differs from the data: none
same prior in OFF and ON: COH (rl), LBR (rl)


,explicit_prior,regular_off_prior,explicit_practice,explicit_trials
subject_id,,,,
CG,gr,rl,1,646
BER,rl,gl,0,658
DP,gr,gr,1,668
FUR,gl,rr,0,640
MAR,gl,rr,1,667
SMI,gl,gr,1,637


**Sessions on several clocks.** The clock ranges of some sessions overlap, which is impossible on one PC, so UCLA ran
on more than one machine (or restarts reset the clock) and the clock cannot order sessions across files, unlike at
Case Western. Overlapping pairs:

In [8]:
spans = blocks[["file", "clock_start_s", "clock_end_s"]].sort_values("clock_start_s").to_numpy()
overlaps = [(a[0], b[0]) for i, a in enumerate(spans) for b in spans[i + 1:] if b[1] < a[2] and a[0].split("/")[0] != b[0].split("/")[0]]
print(f"{len(overlaps)} overlapping pairs of files from different sessions, e.g. {overlaps[:3]}")

18 overlapping pairs of files from different sessions, e.g. [('AJL_off/trial', 'PAM_on/trial'), ('AJL_off/trial', 'MBY_off/practice'), ('AJL_off/trial', 'MBY_off/trial')]


**The old copies.** Each regular session and practice file equals the per-session copy the earlier pipeline (0.00)
read from the top of `raw/` (now `raw_backup_20261019/`), checked on 2026-10-20; the earlier pipeline compiled the
`eq` files as separate equal-prior sessions. The explicit sessions were never compiled before.

## 6. Findings

In [9]:
s = summary.set_index("session_id")
trials_only = s[s.instruction == "implicit"]
lines = [
    f"{len(s)} sessions ({len(trials_only)} regular, {len(explicit)} explicit) of {s.subject_id.nunique()} patients; {len(practice)} eq files compiled as practice.",
    f"Header times: {blocks.header_is_batch_save.sum()} of {len(blocks)} files are batch re-saves "
    f"({', '.join(sorted({f'{t:%Y-%m-%d}' for t in blocks.loc[blocks.header_is_batch_save, 'header_time']}))}); their recording dates are lost.",
    f"Clock runs backwards within a file: {', '.join(blocks.loc[blocks.clock_backwards > 0, 'file']) or 'none'}; pauses over {blk.PAUSE_S} s: median "
    f"{s.n_pauses.median():.0f} per session, longest {s.longest_pause_min.max():.1f} min ({s.longest_pause_min.idxmax()}).",
    f"Prior changes within a session: {', '.join(s.index[~s.prior_consistent]) or 'none'}; design off 75:25 / 50:50: {', '.join(s.index[~s.design_ok]) or 'none'}.",
    f"Practice (eq) before its session on the clock: {int(s.practice_before_trial.eq(True).sum())} of {int(s.n_practice_blocks.gt(0).sum())}; "
    f"gap {s.practice_to_trial_min.min():.1f}-{s.practice_to_trial_min.max():.1f} min. Sessions without practice: {', '.join(s.index[s.n_practice_blocks == 0])}.",
    f"Trials: {s.n_shown.min():.0f}-{s.n_shown.max():.0f} (median {s.n_shown.median():.0f}); under {blk.MIN_SESSION_TRIALS}: {', '.join(s.index[~s.reaches_600]) or 'none'}.",
    f"Valid trials: {s.valid_pct.min():.0f}-{s.valid_pct.max():.0f}%.",
    f"File-name prior differs from the data: {', '.join(label_mismatch.index) or 'none'}. Same prior in OFF and ON: "
    + (", ".join(f"{i} ({p})" for i, p in same_prior.off.items()) or "none") + ".",
    "Explicit sessions: " + "; ".join(f"{i} {r.explicit_prior} (regular OFF {r.regular_off_prior})" for i, r in explicit_table.iterrows()) + ".",
    f"Files from different sessions overlapping on the clock: {len(overlaps)} pairs (several PCs or restarts).",
]
print("\n".join(f"- {line}" for line in lines))

- 40 sessions (34 regular, 6 explicit) of 17 patients; 20 eq files compiled as practice.
- Header times: 60 of 60 files are batch re-saves (2018-05-03, 2022-11-14); their recording dates are lost.
- Clock runs backwards within a file: none; pauses over 20 s: median 0 per session, longest 9.0 min (SMI_off_explicit).
- Prior changes within a session: none; design off 75:25 / 50:50: none.
- Practice (eq) before its session on the clock: 20 of 20; gap 0.9-3.9 min. Sessions without practice: AJL_off, AJL_on, PAM_off, PAM_on, RW_off, RW_on, SKU_off, SKU_on, ALE_off, ALE_on, BBK_off, BBK_on, BER_off, BER_on, DCAM_off, DCAM_on, DMO_off, DMO_on, BER_off_explicit, FUR_off_explicit.
- Trials: 637-760 (median 654); under 600: none.
- Valid trials: 62-100%.
- File-name prior differs from the data: none. Same prior in OFF and ON: COH (rl), LBR (rl).
- Explicit sessions: CG gr (regular OFF rl); BER rl (regular OFF gl); DP gr (regular OFF gr); FUR gl (regular OFF rr); MAR gl (regular OFF rr); SMI gl (

## 7. UCLA healthy controls: the Perugini (2016) key-press controls

All key-press trial-by-trial files (`Reports_HC_Exp2_KeyPress/*_Trial[Bb]yTrial_*.mat`), decoded as 0.01 always has
(`decode_perugini_keypress`). These are processed per-trial tables (an `xyz` matrix), with no clock and no practice,
so each control is one trial block. `in_paper_set` marks the 8 whose files the paper's own analysis folder used
(`asmHC_PDdata_2016_paper_with_RawData/*_asmHCforPD_*.mat`); their trial counts are checked against it. Controls are
named by subject only (`AV`), like the Harvard controls.

In [10]:
paper = PERUGINI / "asmHC_PDdata_2016_paper_with_RawData"
paper_keypress = {f.name.split("_")[0] for f in paper.glob("*_asmHCforPD_*.mat")}
keypress_files = {f.name.split("_")[0]: f for f in sorted((PERUGINI / "Reports_HC_Exp2_KeyPress").glob("*_Trial[Bb]yTrial_*.mat"))}
print(f"key-press controls: {len(keypress_files)} ({', '.join(keypress_files)}); in the paper's set: {len(paper_keypress & set(keypress_files))}; "
      f"not in it: {sorted(set(keypress_files) - paper_keypress)}")

hc_rows, hc_decoded = [], {}
for subject, path in keypress_files.items():
    trials = decode_perugini_keypress(scipy.io.loadmat(path)["xyz"])
    if subject in paper_keypress:
        paper_rt = scipy.io.loadmat(next(paper.glob(f"{subject}_asmHCforPD_*.mat")))["RT"]
        assert len(trials) == len(paper_rt), f"{subject}: {len(trials)} trials, the paper's file has {len(paper_rt)}"
    hc_rows.append({"session_id": subject, "subject_id": subject, "group": "hc", "medication": None, "phase": "trial",
                    "file": f"{subject}/trial", "path": path.relative_to(UCLA).as_posix(), "n_rows": len(trials), "use": True,
                    "valid_pct": 100 * trials.is_valid.mean(), "dataset": "perugini_2016", "response_mode": "keypress",
                    "in_paper_set": subject in paper_keypress})
    hc_decoded[f"{subject}/trial"] = trials
controls = pd.DataFrame(hc_rows)
blk.add_design(controls, hc_decoded)
blk.add_stretch_priors(controls, hc_decoded)

stretch_lists = controls.stretch_priors.str.split(" \\| ")
control_summary = pd.DataFrame({
    "session_id": controls.session_id, "subject_id": controls.subject_id, "group": "hc", "medication": None,
    "n_trials": controls.n_rows, "n_shown": controls.n_shown, "valid_pct": controls.valid_pct, "session_prior": controls.block_prior,
    "stretch_priors": controls.stretch_priors, "prior_consistent": stretch_lists.map(lambda p: len(set(p)) <= 1),
    "max_design_error": controls.design_error, "design_ok": controls.design_error <= blk.DESIGN_TOLERANCE,
    "n_trial_blocks": 1, "n_practice_blocks": 0, "reaches_600": controls.n_shown >= blk.MIN_SESSION_TRIALS,
    "instruction": "implicit", "dataset": "perugini_2016", "response_mode": "keypress", "in_paper_set": controls.in_paper_set,
})
control_summary["issues"] = control_summary.apply(lambda r: "; ".join(
    ([f"prior changes ({r.stretch_priors})"] if not r.prior_consistent else [])
    + ([f"design off 75:25 / 50:50 (error {r.max_design_error:.2f})"] if not r.design_ok else [])
    + ([f"{r.n_shown:.0f} trials"] if not r.reaches_600 else [])
    + ["no practice file"]), axis=1)
control_summary[["session_id", "n_trials", "n_shown", "valid_pct", "session_prior", "stretch_priors", "design_ok", "in_paper_set", "issues"]].round(1)

key-press controls: 10 (AV, BC, BF, EM, ES, GF, GP, JA, MRM, SY); in the paper's set: 8; not in it: ['GF', 'JA']


,session_id,n_trials,n_shown,valid_pct,session_prior,stretch_priors,design_ok,in_paper_set,issues
0,AV,682,680,99.6,gr,gr | gr | gr | gr,True,True,no practice file
1,BC,645,644,98.1,rr,rr | rr | rr | rr,True,True,no practice file
2,BF,836,643,73.8,gr,gr | gr | gr | gr,True,True,no practice file
3,EM,668,666,98.1,gl,gl | gl | gl | gl,True,True,no practice file
4,ES,675,672,98.5,rl,rl | rl | rl | rl,True,True,no practice file
5,GF,988,666,62.8,rr,rr | rr | rr | rr,True,False,no practice file
6,GP,819,673,73.3,gl,gl | gl | gl | gl,True,True,no practice file
7,JA,667,660,93.4,rl,rl | rl | rl | rl,True,False,no practice file
8,MRM,661,642,94.7,rr,rr | rr | rr | rr,True,True,no practice file
9,SY,650,649,98.8,gr,gr | gr | gr | gr,True,True,no practice file


## 8. Compile UCLA: patients and key-press controls

Each patient session as one trial table with the shared columns: the `eq` file (if any) as practice block 1, the
session file as trial block 1 (`block_file` `<session>/practice` or `<session>/trial`). Explicit sessions are separate
sessions (`<patient>_off_explicit`); the summary's `instruction` column marks them. Each key-press control is one trial
block (`<subject>/trial`); `dataset`, `response_mode` and `in_paper_set` are in the summary.

In [11]:
all_blocks = pd.concat([blocks, controls], ignore_index=True)
all_summary = pd.concat([summary, control_summary], ignore_index=True)
blk.compile_sessions(all_blocks, {**decoded, **hc_decoded}, all_summary, "compiled_ucla_session")
catalog.save(all_blocks.drop(columns=blk.INTERNAL_COLUMNS + ["name_label"], errors="ignore"), "compiled_ucla_blocks")
catalog.save(all_summary, "compiled_ucla_session_summary")

saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/AJL_off.parquet


saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/AJL_on.parquet


saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/CG_off.parquet


saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/CG_on.parquet
saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/COH_off.parquet


saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/COH_on.parquet


...

saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/EM.parquet


saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/ES.parquet


saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/GF.parquet


saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/GP.parquet


saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/JA.parquet
saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/MRM.parquet
saved compiled_ucla_session -> /mnt/pd-data/compiled/ucla/sessions/SY.parquet
saved compiled_ucla_blocks -> /mnt/pd-data/compiled/ucla/blocks.csv
saved compiled_ucla_session_summary -> /mnt/pd-data/compiled/ucla/session_summary.csv


## 9. Perugini (2016) eye-movement controls (not used)

The 4 controls who responded by eye movement (`Reports_HC_Exp2/*_rprt1_*.xlsx`), decoded as 0.01 does
(`decode_perugini_eye`), are compiled apart to `compiled/ucla/perugini/` with `dataset` and `response_mode` on every
trial. They are not study data (config `INCLUDE_EYE_TRACKING_SESSIONS: false`) and `catalog.load_block_sessions`
does not read them.

In [12]:
eye_files = {f.name.split("_")[1]: f for f in sorted((PERUGINI / "Reports_HC_Exp2").glob("*_rprt1_*.xlsx"))}
eye_rows, eye_decoded = [], {}
for subject, path in eye_files.items():
    eye_rows.append({"session_id": f"{subject}_perugini_eye", "subject_id": subject, "response_mode": "eye_movement"})
    eye_decoded[f"{subject}_perugini_eye/trial"] = decode_perugini_eye(pd.read_excel(path))
eye = pd.DataFrame(eye_rows).assign(group="hc", medication=None, phase="trial", use=True, clock_start_s=0.0, dataset="perugini_2016")
eye["file"] = eye.session_id + "/trial"
blk.add_design(eye, eye_decoded)
eye["n_trials"] = eye.file.map(lambda f: len(eye_decoded[f]))
eye["valid_pct"] = eye.file.map(lambda f: 100 * eye_decoded[f].is_valid.mean())
eye_summary = eye.rename(columns={"block_prior": "session_prior", "design_error": "max_design_error"}).assign(n_trial_blocks=1)
eye_summary["design_ok"] = eye_summary.max_design_error <= blk.DESIGN_TOLERANCE
eye_summary = eye_summary[["session_id", "subject_id", "group", "medication", "dataset", "response_mode", "n_trial_blocks", "n_trials",
                           "n_shown", "valid_pct", "session_prior", "max_design_error", "design_ok"]]
blk.compile_sessions(eye, eye_decoded, eye_summary, "compiled_perugini_session", extra_columns=("dataset", "response_mode"))
catalog.save(eye_summary, "compiled_perugini_session_summary")
eye_summary.round(1)

saved compiled_perugini_session -> /mnt/pd-data/compiled/ucla/perugini/sessions/CB1_perugini_eye.parquet
saved compiled_perugini_session -> /mnt/pd-data/compiled/ucla/perugini/sessions/JW_perugini_eye.parquet


saved compiled_perugini_session -> /mnt/pd-data/compiled/ucla/perugini/sessions/MEG_perugini_eye.parquet
saved compiled_perugini_session -> /mnt/pd-data/compiled/ucla/perugini/sessions/KR_perugini_eye.parquet
removed stale compiled_perugini_session file AV_perugini_keypress.parquet
removed stale compiled_perugini_session file BC_perugini_keypress.parquet
removed stale compiled_perugini_session file BF_perugini_keypress.parquet
removed stale compiled_perugini_session file EM_perugini_keypress.parquet
removed stale compiled_perugini_session file ES_perugini_keypress.parquet
removed stale compiled_perugini_session file GP_perugini_keypress.parquet
removed stale compiled_perugini_session file MRM_perugini_keypress.parquet
removed stale compiled_perugini_session file SY_perugini_keypress.parquet
saved compiled_perugini_session_summary -> /mnt/pd-data/compiled/ucla/perugini/session_summary.csv


,session_id,subject_id,group,medication,dataset,response_mode,n_trial_blocks,n_trials,n_shown,valid_pct,session_prior,max_design_error,design_ok
0,CB1_perugini_eye,CB1,hc,None,perugini_2016,eye_movement,1,958,624,65.1,rl,0.0,True
1,JW_perugini_eye,JW,hc,None,perugini_2016,eye_movement,1,883,537,60.7,gl,0.0,True
2,MEG_perugini_eye,MEG,hc,None,perugini_2016,eye_movement,1,1084,588,54.2,rr,0.0,True
3,KR_perugini_eye,KR,hc,None,perugini_2016,eye_movement,1,997,457,45.6,gl,0.0,True
